# 01. Preprocessing, Joins, and Rebuilding RQ1/RQ2

This notebook rebuilds the two most important linkages:

1. **Preference -> Consumption:** do users click what they said they prefer?
2. **Exposure -> Consumption:** do clicks come from logged recommendation lists?



<!-- PEC_RERUN_SAFETY_V1 -->

## Execution safety note

This notebook is an authorized private rerun notebook. It can use real processed data when `PEC_GAP_PROCESSED_DIR` or `PEC_GAP_PRIVATE_ROOT` is set, but notebook-generated outputs should stay under `tmp/repro_deep/...` unless you intentionally publish approved aggregate artifacts. The manuscript-facing `reports/tables/`, `reports/figures/`, and Overleaf folders are treated as read-only reference artifacts in this notebook.

**Public-package behavior:** If authorized processed data are unavailable, this notebook skips private-data computation cells without raising an error. This silent skip is expected and does not reproduce manuscript estimates. For a fully runnable bundled-data example, open `../01_metric_walkthrough_synthetic.ipynb` or run `python scripts/run_public_synthetic_pipeline.py` from the repository root.


<!-- PEC_NOTEBOOK_EXPLANATION_V1:joining_guide -->

## Guide: why joins matter

This notebook reconstructs the core PEC linkages. 
The paper's claims depend on whether each click can be matched to metadata, user profile state, and logged recommendation exposure.

The main joins are:

1. **Click -> article metadata:** gives clicked categories and publishers.
2. **User profile -> click history:** compares stated or weighted preferences with clicked consumption.
3. **Recommendation exposure -> click:** checks whether clicked articles appeared in same-day or historical logged recommendation lists.

A failed join is not merely a technical error. It defines an observability boundary. 
For example, a click that cannot be linked to a recommendation list should not be counted as proof that recommendation exposure failed; it may have occurred through another app surface.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:rq1_rq2_interpretation -->

## How to interpret RQ1 and RQ2 outputs

- **RQ1: preference-consumption alignment** asks whether users click categories/publishers that resemble their stated or weighted profile preferences.
- **RQ2: exposure-consumption traceability** asks whether clicked articles can be found in the logged recommendation layer.

Both are descriptive audit quantities. RQ1 is not a psychological measure of true preference. RQ2 is not CTR and not a complete exposure measure. The goal is to separate observed traces before making personalization claims.

In [ ]:
from pathlib import Path
import os
import ast
import json
import math
import sys
import warnings
from collections import Counter


def find_repo_root() -> Path:
    """Find the public/release repository root, independent of private data location."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "reports").exists() and (candidate / "notebooks").exists() and (candidate / "scripts").exists():
            return candidate
    env_repo = os.environ.get("PEC_GAP_REPO_ROOT")
    if env_repo and Path(env_repo).exists():
        return Path(env_repo)
    raise FileNotFoundError("Could not find the PEC repository root. Set PEC_GAP_REPO_ROOT if needed.")


def find_processed_dir(repo_root: Path) -> Path:
    """Find authorized private processed tables without requiring them to live in the repo."""
    exact = os.environ.get("PEC_GAP_PROCESSED_DIR")
    if exact:
        return Path(exact)
    private_root = os.environ.get("PEC_GAP_PRIVATE_ROOT")
    if private_root:
        return Path(private_root) / "data" / "processed"
    local_config = repo_root / ".pecgap_local_paths.json"
    if local_config.exists():
        try:
            cfg = json.loads(local_config.read_text(encoding="utf-8"))
            if cfg.get("processed_dir"):
                return Path(cfg["processed_dir"])
            if cfg.get("private_root"):
                return Path(cfg["private_root"]) / "data" / "processed"
        except Exception as exc:
            print(f"Warning: could not read {local_config}: {exc}")
    return repo_root / "data" / "processed"


ROOT = find_repo_root()
PROCESSED = find_processed_dir(ROOT)
PKG = ROOT / ".python_packages"
if PKG.exists() and sys.version_info[:2] == (3, 12) and str(PKG) not in sys.path:
    # The project package cache is compiled for Python 3.12.
    # Anaconda/Jupyter kernels on this machine are often Python 3.10, so
    # we only add the cache when the ABI matches.
    sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Read-only manuscript/reference artifacts. These are used for comparison.
# For authorized reruns, prefer the private project's reports/tables when the
# processed data live outside this public/release repo. This keeps row-level
# validation tables out of GitHub while still letting the notebooks compare
# regenerated values against the private run artifacts.
PRIVATE_ROOT = PROCESSED.parent.parent if PROCESSED.name == "processed" and PROCESSED.parent.name == "data" else ROOT
PRIVATE_TABLES = PRIVATE_ROOT / "reports" / "tables"
PRIVATE_FIGURES = PRIVATE_ROOT / "reports" / "figures"
PUBLIC_TABLES = ROOT / "reports" / "tables"
PUBLIC_FIGURES = ROOT / "reports" / "figures"
TABLES = PRIVATE_TABLES if PRIVATE_TABLES.exists() else PUBLIC_TABLES
FIGURES = PRIVATE_FIGURES if PRIVATE_FIGURES.exists() else PUBLIC_FIGURES

# Notebook-generated outputs go here by default, not into reports/tables or Overleaf.
NOTEBOOK_STEM = "01_preprocessing_and_joining"
SCRATCH = ROOT / "tmp" / "repro_deep" / NOTEBOOK_STEM
SCRATCH_TABLES = SCRATCH / "tables"
SCRATCH_FIGURES = SCRATCH / "figures"
SCRATCH_TABLES.mkdir(parents=True, exist_ok=True)
SCRATCH_FIGURES.mkdir(parents=True, exist_ok=True)

PRIVATE_DATA_AVAILABLE = PROCESSED.exists()

print(f"Repository root: {ROOT}")
print(f"Processed data: {PROCESSED}")
print(f"Processed data exists: {PRIVATE_DATA_AVAILABLE}")
print(f"Read-only reference tables: {TABLES}")
print(f"Scratch output tables: {SCRATCH_TABLES}")
print(f"Scratch output figures: {SCRATCH_FIGURES}")
if not PRIVATE_DATA_AVAILABLE:
    display(Markdown(
        "**Private processed data not found.** This notebook is real rerun code, "
        "but it needs authorized processed CSVs. Set `PEC_GAP_PROCESSED_DIR` to the exact "
        "processed-data folder, or set `PEC_GAP_PRIVATE_ROOT` to a root containing `data/processed/`. Remaining private-data computation cells will be skipped; use the public synthetic pipeline for a fully runnable bundled-data example."
    ))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    def safe_parse(value):
        '''Parse JSON-like strings stored in CSV cells.''';
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return None
        text = str(value).strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                continue
        return None
    
    
    def truthy(series: pd.Series) -> pd.Series:
        '''Normalize common true-like CSV values to booleans.''';
        return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])
    
    
    def pref_set(row: pd.Series, primary: str, fallback: str) -> set[str]:
        '''Return a stated preference set from explicit selections or weighted profile state.
    
        primary is usually the user-selection column. fallback is the current weighted
        profile-state column. For a weighted dictionary, small positive weights are
        treated as selected dimensions for set-overlap analyses.
        '''
        parsed = safe_parse(row.get(primary, ""))
        if parsed is None:
            parsed = safe_parse(row.get(fallback, ""))
        if isinstance(parsed, dict):
            out = set()
            for key, value in parsed.items():
                try:
                    if float(value) > 0.01:
                        out.add(str(key))
                except Exception:
                    out.add(str(key))
            return out or set(map(str, parsed.keys()))
        if isinstance(parsed, list):
            out = set()
            for item in parsed:
                if isinstance(item, dict) and "value" in item:
                    out.add(str(item["value"]))
                else:
                    out.add(str(item))
            return out
        return set()
    
    
    def normalize_dist(values: dict[str, float]) -> dict[str, float]:
        clean = {}
        for key, value in values.items():
            try:
                val = float(value)
            except Exception:
                continue
            if key and val > 0:
                clean[str(key)] = val
        total = sum(clean.values())
        if total <= 0:
            return {}
        return {key: val / total for key, val in clean.items()}
    
    
    def parse_weight_dist(value) -> dict[str, float]:
        '''Parse a weighted preference dictionary such as {'Politics': 0.7, ...}.''';
        parsed = safe_parse(value)
        if isinstance(parsed, dict):
            return normalize_dist({str(k): v for k, v in parsed.items()})
        if isinstance(parsed, list):
            vals = []
            for item in parsed:
                if isinstance(item, dict) and "value" in item:
                    vals.append(str(item["value"]))
                else:
                    vals.append(str(item))
            if vals:
                return {key: 1 / len(vals) for key in vals if key}
        return {}
    
    
    def norm_entropy(values) -> float:
        '''Normalized entropy in [0, 1]. Higher means more diverse.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        counts = values.value_counts().to_numpy(dtype=float)
        probs = counts / counts.sum()
        ent = -(probs * np.log(probs)).sum()
        max_ent = np.log(len(counts)) if len(counts) > 1 else 1.0
        return float(ent / max_ent) if max_ent else 0.0
    
    
    def norm_entropy_dist(dist: dict[str, float]) -> float:
        values = np.array([v for v in dist.values() if v > 0], dtype=float)
        if len(values) == 0:
            return 0.0
        values = values / values.sum()
        ent = float(-(values * np.log(values)).sum())
        denom = math.log(len(values)) if len(values) > 1 else 1.0
        return ent / denom if denom else 0.0
    
    
    def top_share(values) -> float:
        '''Share of the most common item. Higher means more concentrated.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        return float(values.value_counts().iloc[0] / len(values))
    
    
    def hhi(values) -> float:
        '''Herfindahl-Hirschman concentration index. Higher means more concentrated.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        probs = values.value_counts(normalize=True).to_numpy(dtype=float)
        return float(np.sum(probs * probs))
    
    
    def jaccard(a: set[str], b: set[str]) -> float:
        if not a or not b:
            return np.nan
        return len(a & b) / len(a | b)
    
    
    def distribution_from_values(values) -> dict[str, float]:
        series = pd.Series(values).dropna().astype(str)
        series = series[series != ""]
        if series.empty:
            return {}
        counts = series.value_counts()
        return {str(k): float(v) / float(counts.sum()) for k, v in counts.items()}
    
    
    def cosine(a: dict[str, float], b: dict[str, float]) -> float:
        keys = sorted(set(a) | set(b))
        if not keys:
            return np.nan
        av = np.array([a.get(k, 0.0) for k in keys], dtype=float)
        bv = np.array([b.get(k, 0.0) for k in keys], dtype=float)
        denom = np.linalg.norm(av) * np.linalg.norm(bv)
        return float(np.dot(av, bv) / denom) if denom > 0 else np.nan
    
    
    def js_divergence(a: dict[str, float], b: dict[str, float]) -> float:
        '''Jensen-Shannon divergence between two discrete distributions.''';
        keys = sorted(set(a) | set(b))
        if not keys:
            return np.nan
        p = np.array([a.get(k, 0.0) for k in keys], dtype=float)
        q = np.array([b.get(k, 0.0) for k in keys], dtype=float)
        if p.sum() <= 0 or q.sum() <= 0:
            return np.nan
        p = p / p.sum()
        q = q / q.sum()
        m = 0.5 * (p + q)
        def kl(x, y):
            mask = x > 0
            return float((x[mask] * np.log2(x[mask] / y[mask])).sum())
        return 0.5 * kl(p, m) + 0.5 * kl(q, m)
    
    
    def dist_top_key(dist: dict[str, float]) -> str:
        return max(dist.items(), key=lambda item: item[1])[0] if dist else ""
    
    
    def classify_surface(row: pd.Series) -> str:
        '''Map raw app parameters to the surface categories used in the paper.''';
        source = str(row.get("p_source", "") or "").lower()
        section = str(row.get("section", "") or "").lower()
        message = str(row.get("message_type", "") or "").lower()
        is_headline = str(row.get("is_headline", "") or "").lower() in {"1.0", "1", "true", "yes"}
        if "push" in message or "notification" in source or "notification" in section:
            return "notification"
        if is_headline or "headline" in section:
            return "headline"
        if section.startswith("category_"):
            return "category"
        if "for_you" in section or source == "home":
            return "home/feed"
        if "related" in section or source == "news_detail":
            return "article-detail"
        if "publisher" in section or source in {"newsroom", "source"}:
            return "newsroom"
        if "podcast" in section or "podcast" in source:
            return "podcast"
        if "insight" in section or "insight" in source:
            return "insights"
        if "search" in section or source == "search":
            return "search"
        return "other/unknown"
    
    
    def mask_id(value: str) -> str:
        '''Mask user identifiers before showing examples in a notebook.''';
        text = str(value)
        if len(text) <= 8:
            return text[:2] + "***"
        return text[:4] + "***" + text[-4:]


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    profiles = pd.read_csv(PROCESSED / "user_profiles_clean.csv", dtype=str).fillna("")
    clicks = pd.read_csv(PROCESSED / "click_events_enriched.csv", dtype=str).fillna("")
    recs = pd.read_csv(PROCESSED / "recommendation_items_enriched.csv", dtype=str).fillna("")
    
    matched_clicks = clicks.loc[truthy(clicks["news_matched"])].copy()
    matched_recs = recs.loc[truthy(recs["news_matched"])].copy()
    
    print(f"Profiles: {len(profiles):,}")
    print(f"Matched click rows: {len(matched_clicks):,}")
    print(f"Matched recommendation items: {len(matched_recs):,}")


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Build stated preference sets. These are the set-based inputs used for
    # Jaccard and top-category match analyses.
    profiles = profiles.copy()
    profiles["stated_categories"] = profiles.apply(
        lambda r: pref_set(r, "interested_categories_user_selection", "interested_categories"), axis=1
    )
    profiles["stated_publishers"] = profiles.apply(
        lambda r: pref_set(r, "preferred_news_sources_user_selection", "preferred_news_sources"), axis=1
    )
    profiles["n_stated_categories"] = profiles["stated_categories"].map(len)
    profiles["n_stated_publishers"] = profiles["stated_publishers"].map(len)
    
    display(profiles[["n_stated_categories", "n_stated_publishers"]].describe().T)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Rebuild RQ1 user-level preference-consumption alignment.
    profile_map = profiles.set_index("user_id")
    rows = []
    
    for user_id, group in matched_clicks.groupby("user_id"):
        if user_id not in profile_map.index:
            continue
        profile = profile_map.loc[user_id]
        categories = group["category_final"].replace("", pd.NA).dropna().astype(str).tolist()
        publishers = group["publisher_id_final"].replace("", pd.NA).dropna().astype(str).tolist()
        if not categories:
            continue
    
        top_cat = Counter(categories).most_common(1)[0][0]
        top_pub = Counter(publishers).most_common(1)[0][0] if publishers else ""
        top3_cat = {cat for cat, _ in Counter(categories).most_common(3)}
        top5_pub = {pub for pub, _ in Counter(publishers).most_common(5)}
    
        rows.append({
            "user_id": user_id,
            "masked_user": mask_id(user_id),
            "click_count": len(group),
            "top_clicked_category": top_cat,
            "top_clicked_publisher": top_pub,
            "top_category_in_stated": top_cat in profile["stated_categories"],
            "top_publisher_in_stated": top_pub in profile["stated_publishers"],
            "category_jaccard_top3": jaccard(profile["stated_categories"], top3_cat),
            "publisher_jaccard_top5": jaccard(profile["stated_publishers"], top5_pub),
            "click_category_entropy": norm_entropy(categories),
            "click_publisher_entropy": norm_entropy(publishers),
            "click_top_category_share": top_share(categories),
            "click_top_publisher_share": top_share(publishers),
            "num_stated_categories": len(profile["stated_categories"]),
            "num_stated_publishers": len(profile["stated_publishers"]),
        })
    
    rq1 = pd.DataFrame(rows)
    display(rq1.head(10).drop(columns=["user_id"]))
    display(rq1[[
        "click_count", "category_jaccard_top3", "publisher_jaccard_top5",
        "click_category_entropy", "click_top_category_share",
    ]].describe().T)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    rq1_summary = {
        "users": len(rq1),
        "top_category_match_pct": rq1["top_category_in_stated"].mean() * 100,
        "top_publisher_match_pct": rq1["top_publisher_in_stated"].mean() * 100,
        "mean_category_jaccard_top3": rq1["category_jaccard_top3"].mean(),
        "median_category_jaccard_top3": rq1["category_jaccard_top3"].median(),
    }
    display(pd.DataFrame([rq1_summary]).T.rename(columns={0: "value"}))
    
    saved_rq1 = pd.read_csv(TABLES / "rq1_preference_consumption_gap_by_user.csv")
    comparison = pd.DataFrame([
        {
            "metric": "users",
            "rebuilt": len(rq1),
            "saved_table": len(saved_rq1),
        },
        {
            "metric": "top_category_match_pct",
            "rebuilt": rq1["top_category_in_stated"].mean() * 100,
            "saved_table": saved_rq1["top_category_in_stated"].mean() * 100,
        },
        {
            "metric": "mean_category_jaccard_top3",
            "rebuilt": rq1["category_jaccard_top3"].mean(),
            "saved_table": saved_rq1["category_jaccard_top3"].mean(),
        },
    ])
    display(comparison)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Rebuild RQ2 click-exposure traceability. A click can only be matched to
    # a same-day recommendation when that user's recommendation list was logged
    # for the same date.
    matched_clicks["event_date_str"] = pd.to_datetime(
        matched_clicks["event_date_iso"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")
    
    rec_by_user_date = matched_recs.groupby(["user_id", "recommendation_date"])["news_id"].apply(set).to_dict()
    rec_by_user = matched_recs.groupby("user_id")["news_id"].apply(set).to_dict()
    
    trace_rows = []
    for _, row in matched_clicks.iterrows():
        key = (row["user_id"], row["event_date_str"])
        same_day_set = rec_by_user_date.get(key, set())
        ever_set = rec_by_user.get(row["user_id"], set())
        trace_rows.append({
            "user_id": row["user_id"],
            "event_date": row["event_date_str"],
            "news_id": row["news_id"],
            "category": row["category_final"],
            "publisher_id": row["publisher_id_final"],
            "surface": classify_surface(row),
            "has_same_day_recommendations": bool(same_day_set),
            "clicked_in_same_day_recs": row["news_id"] in same_day_set,
            "clicked_ever_recommended": row["news_id"] in ever_set,
        })
    
    rq2_clicks = pd.DataFrame(trace_rows)
    same_day = rq2_clicks.loc[rq2_clicks["has_same_day_recommendations"]]
    display(pd.DataFrame([{
        "click_rows": len(rq2_clicks),
        "same_day_comparable_rows": len(same_day),
        "same_day_hit_rate": same_day["clicked_in_same_day_recs"].mean(),
        "ever_recommended_hit_rate": rq2_clicks["clicked_ever_recommended"].mean(),
    }]).T.rename(columns={0: "value"}))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # User-level RQ2 table.
    user_rows = []
    for user_id, group in rq2_clicks.groupby("user_id"):
        comparable = group.loc[group["has_same_day_recommendations"]]
        user_rows.append({
            "user_id": user_id,
            "click_rows": len(group),
            "same_day_available_rows": len(comparable),
            "same_day_hit_rows": int(comparable["clicked_in_same_day_recs"].sum()),
            "same_day_hit_rate": comparable["clicked_in_same_day_recs"].mean() if len(comparable) else np.nan,
            "ever_recommended_hit_rate": group["clicked_ever_recommended"].mean(),
        })
    rq2_users = pd.DataFrame(user_rows)
    display(rq2_users.describe().T)
    
    surface_summary = rq2_clicks.groupby("surface").agg(
        click_rows=("news_id", "size"),
        same_day_hit_rate=("clicked_in_same_day_recs", "mean"),
        ever_recommended_hit_rate=("clicked_ever_recommended", "mean"),
    ).assign(click_share=lambda d: d["click_rows"] / d["click_rows"].sum()).sort_values("click_rows", ascending=False)
    display(surface_summary)


**Checkpoint**

At this point you have rebuilt the two basic audit linkages. The next
notebook turns these rows into the paper's PEC metrics and robustness
checks.


## Step map: what Notebook 01 rebuilds

This notebook centers on two joins.

First, the **profile → clicks** join compares each user's stated category/source sets with their clicked category/source sets to construct RQ1 P-C alignment.

Second, the **clicks → recommendation exposure** join tests whether a clicked article `news_id` appeared in the same user's recommendation list on the same date, constructing RQ2 E-C traceability.

Both can look like routine joins, but their success or failure defines an observability boundary. A click absent from a logged recommendation list is not necessarily a recommendation failure; it may be click consumption that the logged list alone cannot trace.


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    notebook01_flow = pd.DataFrame([
        {
            "stage": "1. Load processed tables",
            "input": "profiles, click_events_enriched, recommendation_items_enriched",
            "key operation": "filter to metadata-matched click/recommendation rows",
            "why it matters": "article category/source labels are required to compare P/E/C in a common label space.",
            "main variables": "profiles, matched_clicks, matched_recs",
        },
        {
            "stage": "2. Build preference sets",
            "input": "interested_categories_user_selection, preferred_news_sources_user_selection; fallback weighted profile fields",
            "key operation": "parse JSON-like strings into Python sets",
            "why it matters": "Jaccard and top-category match require set representations.",
            "main variables": "stated_categories, stated_publishers",
        },
        {
            "stage": "3. Aggregate clicked consumption by user",
            "input": "matched_clicks grouped by user_id",
            "key operation": "top clicked category, top-3 clicked categories, clicked publishers, entropy summaries",
            "why it matters": "RQ1 compares preferences with a user-level consumption profile, not with isolated click rows.",
            "main variables": "rq1",
        },
        {
            "stage": "4. Match clicks to same-day recommendation sets",
            "input": "matched_clicks + matched_recs grouped by user_id/date",
            "key operation": "check clicked news_id membership in same-day recommendation set and six-month user recommendation history",
            "why it matters": "RQ2 is not CTR; it tests whether a clicked article can be traced in the logged-exposure layer.",
            "main variables": "rq2_clicks, rq2_users, surface_summary",
        },
    ])
    display(notebook01_flow)


## What to check after running Notebook 01

- `rq1` is the user-level P-C alignment table. It produces the 698-user denominator, empty stated-category handling, top-category match, and Jaccard overlap.
- `rq2_clicks` is the event-level traceability table. Each click is marked for same-day recommendation-set membership and six-month recommendation-history membership.
- `rq2_users` summarizes those click events at the user level.
- `surface_summary` shows the app-surface pathways through which clicks arrived.

Notebook 01 does not ask readers to accept the paper's numbers on faith. It shows how row-level joins become the user-level and click-level analysis tables used in RQ1 and RQ2.


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # AUTO-SAVE REPRO_DEEP SCRATCH OUTPUTS
    # This cell writes notebook-generated rerun outputs only to tmp/repro_deep/...
    # It does not write to reports/tables, reports/figures, or the Overleaf submission folder.
    _save_names = ['rq1', 'rq1_summary', 'comparison', 'rq2_clicks', 'rq2_users', 'surface_summary', 'notebook01_flow']
    _saved_paths = []
    for _name in _save_names:
        if _name not in globals():
            continue
        _obj = globals()[_name]
        _path = SCRATCH_TABLES / f"{_name}.csv"
        try:
            if isinstance(_obj, pd.DataFrame):
                _obj.to_csv(_path, index=False)
            elif isinstance(_obj, pd.Series):
                _obj.to_frame().to_csv(_path)
            elif isinstance(_obj, list):
                pd.DataFrame(_obj).to_csv(_path, index=False)
            elif isinstance(_obj, dict):
                pd.DataFrame([_obj]).to_csv(_path, index=False)
            else:
                continue
            _saved_paths.append(_path)
        except Exception as exc:
            print(f"Could not save {_name}: {exc}")
    
    try:
        import matplotlib.pyplot as _plt
        for _idx, _num in enumerate(_plt.get_fignums(), 1):
            _fig = _plt.figure(_num)
            _fig_path = SCRATCH_FIGURES / f"{NOTEBOOK_STEM}_figure_{_idx:02d}.png"
            _fig.savefig(_fig_path, dpi=160, bbox_inches="tight")
            _saved_paths.append(_fig_path)
    except Exception as exc:
        print(f"Could not save open matplotlib figures: {exc}")
    
    print("Saved scratch artifacts:")
    for _path in _saved_paths:
        print(" -", _path)
